___
# <center>Atividade: Encadear Operações</center>
___

## Aula 04

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * escrever uma análise como uma sequência de operações encadeadas;
 * usar os cinco verbos: filtrar, criar coluna, ordenar, escolher colunas e agregar;
 * agrupar uma base e calcular estatísticas por grupo;
 * reconhecer os erros mais comuns do encadeamento.

Até aqui, cada operação foi uma linha separada, guardando o resultado numa
variável nova. Funciona, e fica ilegível rápido.

Nesta aula toda análise vira uma **sequência de operações encadeadas**, escrita
de cima para baixo dentro de um par de parênteses. Cada linha faz uma coisa, e
você lê o que aconteceu na ordem em que aconteceu.


___
<div id="indice"></div>

## Índice

- [Reincidência e regime inicial](#problema)

- [O problema que o encadeamento resolve](#problema-encadeamento)
    - [🔗 A mesma coisa, encadeada](#encadeada)
    - [( ) Por que os parênteses](#parenteses)

- [Os cinco verbos](#verbos)
    - [1️⃣ .query(): escolher linhas](#query)
    - [2️⃣ .assign(): criar colunas](#assign)
    - [3️⃣ .sort_values(): ordenar](#sort)
    - [4️⃣ [[...]]: escolher colunas](#colunas)
    - [5️⃣ .groupby() e .agg(): agregar por grupo](#groupby)

- [Montando o pipeline](#pipeline)
    - [EXERCÍCIO 1: resumo por câmara](#ex1)

- [Dois erros que você vai cometer](#erros)

- [Exercícios](#exercicios)
    - [EXERCÍCIO 2: pena por regime](#ex2)
    - [EXERCÍCIO 3: as cinco maiores comarcas](#ex3)

- [RESUMO](#resumo)


___
<div id="problema"></div>

# Reincidência e regime inicial

A pergunta de hoje:

> Nas apelações criminais do TJSP, a proporção de acórdãos que mencionam
> reincidência varia conforme o regime inicial fixado?

**As variáveis da base:**

* `processo`, `cd_acordao`: identificadores.
* `classe`, `assunto`, `relator`, `comarca`, `orgao_julgador`, `camara`: dados do julgamento.
* `data_julgamento`, `data_publicacao`: datas.
* `regime_inicial`: aberto, semiaberto ou fechado, lido da ementa.
* `pena_anos`: pena em anos, lida da ementa.
* `houve_reincidencia`, `houve_confissao`, `eh_trafico`: indicadores lidos da ementa.
* `n_palavras_ementa`: tamanho da ementa.

Coletada com a biblioteca
[juscraper](https://github.com/jtrecenti/juscraper). **Não precisa rodar:**

```python
import juscraper as jus

tjsp = jus.scraper("tjsp")
acordaos = tjsp.cjsg('"apelacao criminal" E "regime inicial"', paginas=range(1, 26))
```


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


In [ ]:
criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")
criminal.head(3)


In [ ]:
criminal.info()


> ⚠️ `regime_inicial` e `pena_anos` foram lidos do texto da ementa, e nenhum dos
> dois vem completo: o regime aparece em cerca de 70% dos acórdãos e a pena em
> 45%. `pena_anos` ainda traz valores implausíveis, porque a leitura pega o
> primeiro número seguido de "anos" que encontra. Vamos lidar com isso.


[Volta ao Índice](#indice)


___
<div id="problema-encadeamento"></div>

# O problema que o encadeamento resolve

Resolvendo a pergunta do jeito que fizemos até agora, com uma variável nova por
operação:


In [ ]:
apelacoes = criminal[criminal["classe"] == "Apelação Criminal"]
com_regime = apelacoes.dropna(subset=["regime_inicial"])
fechado = com_regime[com_regime["regime_inicial"] == "fechado"]
semiaberto = com_regime[com_regime["regime_inicial"] == "semiaberto"]
aberto = com_regime[com_regime["regime_inicial"] == "aberto"]

pd.Series({
    "fechado": fechado["houve_reincidencia"].mean(),
    "semiaberto": semiaberto["houve_reincidencia"].mean(),
    "aberto": aberto["houve_reincidencia"].mean(),
}).round(3)


Funciona, e tem três problemas:

1. **seis variáveis** que existem só para chegar num resultado, e que continuam
   ocupando memória e atrapalhando a leitura do resto do notebook;
2. **nomes intermediários** como `com_regime` que não querem dizer nada e que
   você vai reaproveitar por engano daqui a três células;
3. **não escala**: se aparecesse um quarto regime, seria preciso escrever mais
   uma linha e lembrar de incluí-la no resultado.


<div id="encadeada"></div>

### 🔗 A mesma coisa, encadeada


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .groupby("regime_inicial")
    .agg(proporcao=("houve_reincidencia", "mean"))
    .round(3)
)


Leia de cima para baixo: pegue `criminal`, fique só com as apelações, descarte
quem não tem regime, junte por regime, e calcule a proporção. Nenhuma variável
intermediária, e a ordem das operações é a ordem das linhas.


<div id="parenteses"></div>

### ( ) Por que os parênteses

Em Python, dentro de um par de parênteses você pode quebrar a linha à vontade.
Sem eles, `criminal` seguido de uma quebra de linha e `.query(...)` é erro de
sintaxe. Os parênteses existem só para deixar você pôr uma operação por linha.

O formato que vamos usar sempre é este:

```python
resultado = (
    tabela
    .operacao_1(...)
    .operacao_2(...)
)
```

Abre parêntese, o nome da tabela sozinho na primeira linha, e daí em diante uma
operação por linha, cada uma começando com ponto.


[Volta ao Índice](#indice)


___
<div id="verbos"></div>

# Os cinco verbos

Quase toda análise descritiva é uma combinação de cinco operações. Vamos uma a
uma.


<div id="query"></div>

### 1️⃣ .query(): escolher linhas


Recebe a condição escrita como **texto**. Dentro das aspas, os nomes das colunas aparecem sem `df[...]`, e o texto que você compara vai entre aspas simples.

✔️ **Uso do `.query()`**

```python
# Sintaxe geral:
DataFrame.query("coluna == 'valor'")
```

Documentação oficial: [.query()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html)


In [ ]:
criminal.query("regime_inicial == 'fechado'").shape


Para combinar condições, use `and`, `or` e `not`, por extenso:


In [ ]:
criminal.query("regime_inicial == 'fechado' and houve_reincidencia").shape


**✍️ Agora você.** Fique só com os acórdãos de tráfico em que houve confissão.


In [ ]:
criminal.query("eh_trafico ________ houve_confissao").shape


Para usar uma variável do Python dentro da condição, ponha `@` na frente dela:


In [ ]:
regime_alvo = "semiaberto"

criminal.query("regime_inicial == @regime_alvo").shape


<div id="assign"></div>

### 2️⃣ .assign(): criar colunas


Devolve uma **cópia** da tabela com a coluna nova. Não altera a tabela original, e é por isso que serve para encadear.

✔️ **Uso do `.assign()`**

```python
# Sintaxe geral:
DataFrame.assign(nome_novo=lambda d: d["coluna"] * 2)
```

Documentação oficial: [.assign()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)


O `lambda d:` quer dizer "a tabela como ela está **neste ponto** da sequência".
O `d` é só um nome, e podia ser qualquer outro.

> ⚠️ Escrever `criminal["coluna"]` dentro do encadeamento estraga tudo: se antes
> do `.assign` houve um filtro, `criminal` ainda é a tabela inteira, as duas não
> têm mais o mesmo número de linhas, e o pandas casa os valores pelo índice, em
> silêncio. Dentro de um encadeamento, **sempre** `lambda d:`.


In [ ]:
(
    criminal
    .query("regime_inicial == 'fechado'")
    .assign(ementa_longa=lambda d: d["n_palavras_ementa"] > 200)
    [["processo", "regime_inicial", "n_palavras_ementa", "ementa_longa"]]
    .head(3)
)


**✍️ Agora você.** Crie a coluna `pena_alta`, verdadeira quando `pena_anos` for maior que 8, usando `lambda`.


In [ ]:
(
    criminal
    .assign(pena_alta=lambda d: d["________"] > 8)
    [["processo", "pena_anos", "pena_alta"]]
    .head(3)
)


Dá para criar várias colunas de uma vez, separando por vírgula. E uma coluna
criada num `.assign` pode ser usada na seguinte, desde que seja com `lambda`:


In [ ]:
(
    criminal
    .assign(
        pena_meses=lambda d: d["pena_anos"] * 12,
        pena_meses_arredondada=lambda d: d["pena_meses"].round(0),
    )
    [["processo", "pena_anos", "pena_meses", "pena_meses_arredondada"]]
    .head(3)
)


<div id="sort"></div>

### 3️⃣ .sort_values(): ordenar


O primeiro argumento diz por qual coluna ordenar, e `ascending=False` inverte para o maior primeiro.

✔️ **Uso do `.sort_values()`**

```python
# Sintaxe geral:
DataFrame.sort_values("coluna", ascending=False)
```

Documentação oficial: [.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)


In [ ]:
(
    criminal
    .sort_values("n_palavras_ementa", ascending=False)
    [["processo", "comarca", "n_palavras_ementa"]]
    .head(5)
)


**✍️ Agora você.** Ordene pela pena, da maior para a menor, e olhe as cinco primeiras. Repare no que aparece: a leitura automática da pena erra em alguns acórdãos.


In [ ]:
(
    criminal
    .sort_values("________", ascending=________)
    [["processo", "pena_anos", "regime_inicial"]]
    .head(5)
)


<div id="colunas"></div>

### 4️⃣ [[...]]: escolher colunas

Duas chaves com uma lista de nomes dentro devolvem só aquelas colunas, na ordem
que você pediu. Já apareceu nos exemplos acima.


In [ ]:
(
    criminal
    [["processo", "comarca", "regime_inicial", "pena_anos"]]
    .head(3)
)


<div id="groupby"></div>

### 5️⃣ .groupby() e .agg(): agregar por grupo

Esta é a operação nova de verdade. `.groupby("coluna")` separa a tabela em
pedaços, um por valor da coluna, e `.agg(...)` calcula uma estatística em cada
pedaço, devolvendo **uma linha por grupo**.


As estatísticas são as mesmas da aula 3, escritas como texto: `"mean"`, `"median"`, `"std"`, `"min"`, `"max"`, `"sum"`, `"nunique"`, além de `"size"`, que conta as linhas do grupo.

✔️ **Uso do `.groupby().agg()`**

```python
# Sintaxe geral:
DataFrame
    .groupby("coluna_de_grupo")
    .agg(nome_da_saida=("coluna_de_entrada", "estatistica"))
    .reset_index()
```

Documentação oficial: [.groupby().agg()](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        mediana_palavras=("n_palavras_ementa", "median"),
    )
)


Repare que `regime_inicial` saiu **fora** da tabela, à esquerda, em negrito:
depois de um `groupby`, a coluna de agrupamento vira o **índice** do resultado,
e não uma coluna normal. Isso atrapalha se você quiser continuar encadeando.

O `.reset_index()` traz o índice de volta para dentro da tabela. Por isso ele
aparece no fim de quase todo `groupby`: com o índice de volta, dá para filtrar e
ordenar o resultado como qualquer outra tabela.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(n=("processo", "size"))
    .reset_index()
)


E vale lembrar da aula 3: a média de uma coluna de verdadeiro e falso é a
proporção. Isso funciona igual dentro do `.agg`.


**✍️ Agora você.** Acrescente ao resumo a proporção de acórdãos com reincidência e a proporção de tráfico.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "________"),
        prop_trafico=("________", "mean"),
    )
    .reset_index()
    .round(3)
)


[Volta ao Índice](#indice)


___
<div id="pipeline"></div>

# Montando o pipeline

Voltando à pergunta: a proporção de menção a reincidência varia conforme o
regime inicial?

Duas coisas ainda faltam. Primeiro, o regime é **ordinal**, e queremos a tabela
na ordem aberto, semiaberto, fechado, e não em ordem alfabética. Isso é a
categórica ordenada da aula 2, criada aqui dentro do `.assign`. Segundo,
`groupby` sobre categórica traz todas as categorias declaradas, e
`observed=True` mantém só as que aparecem.


In [ ]:
resumo = (
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .assign(
        regime=lambda d: pd.Categorical(
            d["regime_inicial"],
            categories=["aberto", "semiaberto", "fechado"],
            ordered=True,
        )
    )
    .groupby("regime", observed=True)
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "mean"),
        prop_confissao=("houve_confissao", "mean"),
        prop_trafico=("eh_trafico", "mean"),
    )
    .round(3)
)

resumo


A leitura é direta: a menção a reincidência sobe conforme o regime fica mais
severo. Isso não é surpresa, é quase a definição legal do regime, e serve para
conferir que a leitura das variáveis está coerente.

> 🤔 E o que **não** dá para concluir: nada sobre causalidade, e nada sobre
> acórdãos em que o regime não foi identificado, que são cerca de 30% da base.


<div id="ex1"></div>

### EXERCÍCIO 1

Monte um resumo parecido, agora por `camara`, mantendo só as câmaras com pelo
menos 15 acórdãos e ordenando da maior proporção de reincidência para a menor.
Você vai precisar de `.reset_index()`, `.query()` e `.sort_values()` depois do
`.agg()`.


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["camara"])
    .groupby("________")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "________"),
    )
    .________()
    .query("n >= ________")
    .sort_values("________", ascending=False)
    .round(3)
)


[Volta ao Índice](#indice)


___
<div id="erros"></div>

# Dois erros que você vai cometer

**1. Esquecer o parêntese de abertura.** Sem os parênteses, a quebra de linha
encerra o comando:


In [ ]:
codigo = '''
criminal
.query("eh_trafico")
'''

try:
    exec(codigo)
except SyntaxError as erro:
    print("SyntaxError:", erro)


**2. Achar que `.assign` altera a tabela.** Ele devolve uma cópia. Se você não
guardar o resultado, a coluna não existe fora do encadeamento:


In [ ]:
criminal.assign(teste=1)

"teste" in criminal.columns


> 🤔 E uma recomendação, que não é erro de sintaxe: sequência com quinze
> operações é tão ruim de ler quanto quinze variáveis soltas. Quando o
> encadeamento passar de umas oito linhas, quebre em duas partes com um nome que
> signifique alguma coisa, como fizemos com `resumo`.


[Volta ao Índice](#indice)


___
<div id="exercicios"></div>

# Exercícios


<div id="ex2"></div>

### EXERCÍCIO 2

A pena lida da ementa tem valores implausíveis, como penas acima de 40 anos, que
vêm de a leitura pegar um número errado. Monte um encadeamento que descarte as
penas ausentes e as maiores que 30 anos, e devolva mediana, média e desvio
padrão da pena por regime inicial.


In [ ]:
(
    criminal
    .dropna(subset=["pena_anos", "regime_inicial"])
    .query("pena_anos ________ 30")
    .groupby("________")
    .agg(
        n=("processo", "size"),
        mediana=("pena_anos", "________"),
        media=("pena_anos", "mean"),
        desvio=("pena_anos", "________"),
    )
    .reset_index()
    .round(2)
)


<div id="ex3"></div>

### EXERCÍCIO 3

Quais são as cinco comarcas com mais apelações criminais nesta base, e qual a
proporção de tráfico em cada uma?


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .groupby("________")
    .agg(n=("processo", "size"), prop_trafico=("eh_trafico", "________"))
    .reset_index()
    .sort_values("________", ascending=False)
    .head(________)
    .round(3)
)


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

Uma análise descritiva é uma sequência de operações, escrita de cima para baixo
dentro de um par de parênteses, com uma operação por linha.

| verbo | para quê |
|---|---|
| `.query("...")` | escolher linhas por uma condição |
| `.dropna(subset=[...])` | descartar linhas sem valor numa coluna |
| `.assign(nova=lambda d: ...)` | criar coluna |
| `[["a", "b"]]` | escolher colunas |
| `.sort_values("a", ascending=False)` | ordenar |
| `.groupby("a").agg(saida=("b", "mean"))` | uma linha por grupo |
| `.reset_index()` | tirar o agrupamento do índice |
| `.head(n)` | cortar as primeiras linhas |


In [ ]:
#=> O FORMATO: abre parêntese, tabela sozinha, uma operação por linha
resumo = (
    criminal

    #=> ESCOLHER LINHAS: condição como texto, colunas sem aspas
    .query("classe == 'Apelação Criminal'")

    #=> DESCARTAR FALTANTES de uma coluna
    .dropna(subset=["regime_inicial"])

    #=> CRIAR COLUNA: lambda d é "a tabela neste ponto da sequência"
    .assign(
        regime=lambda d: pd.Categorical(
            d["regime_inicial"],
            categories=["aberto", "semiaberto", "fechado"],
            ordered=True,
        )
    )

    #=> AGRUPAR: observed=True descarta categorias sem nenhuma linha
    .groupby("regime", observed=True)

    #=> AGREGAR: saida=("coluna_de_entrada", "estatistica")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "mean"),
    )

    #=> TIRAR O AGRUPAMENTO DO ÍNDICE, para poder continuar encadeando
    .reset_index()

    #=> ORDENAR e ARREDONDAR
    .sort_values("prop_reincidencia", ascending=False)
    .round(3)
)

resumo


**Duas regras que valem sempre:**

1. Dentro do encadeamento, olhe para as colunas com `lambda d:`, nunca pelo nome
   da tabela original. Sem isso, o pandas casa os valores pelo índice e erra em
   silêncio.
2. Quando a sequência ficar longa demais para caber na tela, quebre em duas.


[Volta ao Índice](#indice)
